<h1>Trabalho 2 de ARQ - G13</h1>

<h1>Atividade 02 - melhorar o desempenho de RP em conjunto de dados existentes</h1>
<p>A atividade 02 visa trabalhar com um conjunto de dados pré-construído, onde as opções que o desenvolvedor tem, são de aplicar as técnicas de pré-processamento abaixo relacionadas:</p>
<ul><li>Seleção</li>
<li>Limpeza</li>
<li>Codificação</li>
<li>Enriquecimento</li>
<li>Normalização</li>
<li>Construção de Atributos</li>
<li>Correção de Prevalência</li>
<li>Partição do Conjunto de Dados</li>
</ul>
<p>Busque uma base de dados na UCI Machine Learning que seja indicada para problemas de classificação. (<a target="_blank" href="https://archive.ics.uci.edu/datasets">https://archive.ics.uci.edu/datasets</a>)</p>



<p>Nossa opção foi pela base "Adult Census Income", cujo objetivo é predizer se o valor de income é maior que 50k</p>

## 01 - Carregar a base a partir da URL


In [ ]:
import pandas as pd

url_train = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
url_test  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

colunas = [
    "age", "workclass", "fnlwgt", "education",
    "education_num", "marital_status", "occupation",
    "relationship", "race", "sex",
    "capital_gain", "capital_loss",
    "hours_per_week", "native_country",
    "income"
]

train = pd.read_csv(
    url_train,
    header=None,
    names=colunas,
    skipinitialspace=True,
    na_values='?'
)

test = pd.read_csv(
    url_test,
    header=None,
    names=colunas,
    skipinitialspace=True,
    na_values='?'
)

# remove ponto final da classe no teste
test["income"] = test["income"].str.replace('.', '', regex=False)

# concatena
df_raw = pd.concat([train, test], ignore_index=True)


In [2]:
# Mantem o original para facilitar eventauis correções
df = df_raw.copy()

print(df.shape)

(48843, 15)


## 02 - Análise exploratória de dados

In [3]:
print(f'\nDimensões do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas\n')

display("Lista das colunas e tipos:",df.dtypes)


Dimensões do dataset: 48843 linhas x 15 colunas



'Lista das colunas e tipos:'

age                object
workclass             str
fnlwgt            float64
education             str
education_num     float64
marital_status        str
occupation            str
relationship          str
race                  str
sex                   str
capital_gain      float64
capital_loss      float64
hours_per_week    float64
native_country        str
income                str
dtype: object

In [4]:
print("=== Valores faltantes por coluna ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Valores Faltantes': missing, 'Percentual (%)': missing_pct})
print(missing_df[missing_df['Valores Faltantes'] > 0])

=== Valores faltantes por coluna ===
                Valores Faltantes  Percentual (%)
workclass                    2800            5.73
fnlwgt                          1            0.00
education                       1            0.00
education_num                   1            0.00
marital_status                  1            0.00
occupation                   2810            5.75
relationship                    1            0.00
race                            1            0.00
sex                             1            0.00
capital_gain                    1            0.00
capital_loss                    1            0.00
hours_per_week                  1            0.00
native_country                858            1.76
income                          1            0.00


## 03 - Limpeza

In [5]:
# Tratativa: Remoção das linhas com valores em branco
print(f'=== Verificação de linhas com NaN ===')
print(f'\nLinhas antes da limpeza: {len(df)}')
df = df.dropna(how='any')
print(f'Linhas depois da limpeza: {len(df)}')

=== Verificação de linhas com NaN ===

Linhas antes da limpeza: 48843
Linhas depois da limpeza: 45222


In [6]:
# Verifica se existe linhas duplicadas
print(f'=== Verificação de Dados Duplicados ===')

duplicadas = df.duplicated().sum()
print(f'Total de linhas duplicadas: {duplicadas:,}')

# remoção de linhas duplicadas
if duplicadas > 0:
  print("\nIndice das linhas duplicadas:")
  print(df[df.duplicated(keep=False)].index.values)
  df = df.drop_duplicates()
  print(f'\nLinhas após remoção de duplicatas: {len(df):}')
else:
  print('\nNenhuma linha duplicata encontrada. Nenhuma ação necessária.')

=== Verificação de Dados Duplicados ===
Total de linhas duplicadas: 28

Indice das linhas duplicadas:
[ 2303  3917  4325  4767  4881  4940  5104  5579  5805  5842  6990  7053
  7920  8080  8679  9171 10367 11631 11965 13084 15059 15189 16297 16846
 17040 17673 17916 18698 21103 21318 21490 21875 22300 22367 22494 25624
 25872 26313 28230 28522 28846 29157 30845 31993 32404 33050 33426 33881
 36462 39583 41811 43751 43774 46410 48522]

Linhas após remoção de duplicatas: 45194


## 04 - Codificação

In [7]:
# Corrige a coluna 'age' para numerica
df['age'] = df['age'].astype("int64")

In [8]:
# Converte as colunas onde os valores só podem assumir dois possíveis valores
df['sex'] = df['sex'].map({
    "Male": 1,
    "Female": 0
})

df['income'] = df['income'].map({
    "<=50K": 0,
    ">50K": 1
})

In [9]:
# A coluna 'education' pode ser removida em favor da coluna 'education_num' que representa o número de anos de educação
df = df.drop(['education'], axis=1)

In [10]:
# Verifica a quantidades de classes das colunas descritivas
df.select_dtypes(include="str").nunique()

workclass          7
marital_status     7
occupation        14
relationship       6
race               5
native_country    41
dtype: int64

In [11]:
# obtem lista de colunas numericas
colunas_numericas = df.select_dtypes(include="number").columns.tolist()

# Converte colunas categoricas em numericas
workclass_num = pd.DataFrame({'workclass':df['workclass'].astype('category').cat.codes})
marital_status_num = pd.DataFrame({'marital_status':df['marital_status'].astype('category').cat.codes})
occupation_num = pd.DataFrame({'occupation':df['occupation'].astype('category').cat.codes})
relationship_num = pd.DataFrame({'relationship':df['relationship'].astype('category').cat.codes})
race_num = pd.DataFrame({'race':df['race'].astype('category').cat.codes})
native_country_num = pd.DataFrame({'native_country':df['native_country'].astype('category').cat.codes})

df = df.drop(['workclass', 'marital_status', 'occupation','relationship', 'race', 'native_country'], axis=1)

# coluna todas as colunas numericas no mesmo dataframe
df = pd.concat([df[colunas_numericas], 
                workclass_num, marital_status_num, occupation_num, 
                relationship_num, race_num,
                native_country_num],axis=1)

In [12]:
# Separa features de target
y = df['income']
X_orig = df.drop(['income'], axis=1)

## 05 - Partição dos dados originais

In [13]:
from sklearn.model_selection import train_test_split

# Vamos particionar o dataset em 80% para treinamento e 20% para teste
X_treino_orig, X_teste_orig, y_treino_orig, y_teste_orig = train_test_split(
    X_orig, y, test_size=0.2, random_state=13
)

## 06 - Modelo

In [14]:
from sklearn import svm
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

model = svm.SVC()  #algoritmo escolhido


In [15]:
# Treino do modelo sem tratamento
model.fit(X_treino_orig, y_treino_orig)

# predição com os mesmos dados usados para treinar
y_hat_treino_orig = model.predict(X_treino_orig)

cm_orig_train = confusion_matrix(y_treino_orig, y_hat_treino_orig)
print('Matriz de confusão - com os dados ORIGINAIS usados no TREINAMENTO')
print(cm_orig_train)
print(classification_report(y_treino_orig, y_hat_treino_orig))

# predição com os mesmos dados usados para testar
print('Matriz de confusão - com os dados ORIGINAIS usados para TESTES')
y_hat_teste_orig = model.predict(X_teste_orig)
cm_orig_test = confusion_matrix(y_teste_orig, y_hat_teste_orig)
print(cm_orig_test)
print(classification_report(y_teste_orig, y_hat_teste_orig))

Matriz de confusão - com os dados ORIGINAIS usados no TREINAMENTO
[[27129    62]
 [ 7496  1468]]
              precision    recall  f1-score   support

           0       0.78      1.00      0.88     27191
           1       0.96      0.16      0.28      8964

    accuracy                           0.79     36155
   macro avg       0.87      0.58      0.58     36155
weighted avg       0.83      0.79      0.73     36155

Matriz de confusão - com os dados ORIGINAIS usados para TESTES
[[6787   10]
 [1888  354]]
              precision    recall  f1-score   support

           0       0.78      1.00      0.88      6797
           1       0.97      0.16      0.27      2242

    accuracy                           0.79      9039
   macro avg       0.88      0.58      0.57      9039
weighted avg       0.83      0.79      0.73      9039



## 07 - Normalização

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [17]:
# Vamos listar as colunas que necessitam padronizar (contínuas) e as que não devem ser padronizadas (binárias)
col_continuas = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week', 
                 'workclass', 'marital_status', 'occupation', 'relationship', 'race', 'native_country']
col_binarias = ['sex']

In [18]:
# Cria um preprocessamento para transformar apenas as variáveis contínuas e passar as binárias
processamento_X = ColumnTransformer(
    transformers=[
        ("cont", StandardScaler(), col_continuas),
        ("bin", "passthrough", col_binarias)
    ]
)


In [19]:
X_transformed = processamento_X.fit_transform(X_orig)

In [20]:
# Vamos particionar o dataset em 80% para treinamento e 20% para teste
X_treino_norm, X_teste_norm, y_treino_norm, y_teste_norm = train_test_split(
    X_transformed, y, test_size=0.2, random_state=13
)

## 08 - Modelo com dados normalizados

In [21]:
# Treino do modelo com dados normalizados
model.fit(X_treino_norm, y_treino_norm)

# predição com os mesmos dados usados para treinar
y_hat_treino_norm = model.predict(X_treino_norm)

cm_norm_train = confusion_matrix(y_treino_norm, y_hat_treino_norm)
print('Matriz de confusão - com os dados NORMALIZADOS usados no TREINAMENTO')
print(cm_norm_train)
print(classification_report(y_treino_norm, y_hat_treino_norm))

# predição com os mesmos dados usados para testar
print('Matriz de confusão - com os dados NORMALIZADOS usados para TESTES')
y_hat_teste_norm = model.predict(X_teste_norm)
cm_norm_test = confusion_matrix(y_teste_norm, y_hat_teste_norm)
print(cm_norm_test)
print(classification_report(y_teste_norm, y_hat_teste_norm))

Matriz de confusão - com os dados NORMALIZADOS usados no TREINAMENTO
[[25815  1376]
 [ 3934  5030]]
              precision    recall  f1-score   support

           0       0.87      0.95      0.91     27191
           1       0.79      0.56      0.65      8964

    accuracy                           0.85     36155
   macro avg       0.83      0.76      0.78     36155
weighted avg       0.85      0.85      0.84     36155

Matriz de confusão - com os dados NORMALIZADOS usados para TESTES
[[6391  406]
 [1024 1218]]
              precision    recall  f1-score   support

           0       0.86      0.94      0.90      6797
           1       0.75      0.54      0.63      2242

    accuracy                           0.84      9039
   macro avg       0.81      0.74      0.76      9039
weighted avg       0.83      0.84      0.83      9039

